# Algoritmo Genético para el TSP
**Asignatura:** Algoritmos de Optimización

**Alumno:** *Ricardo Germes Serrano* <br>

**URL Repositorio GitHub de las 3 Prácticas:** *https://github.com/RGS70/03MIAR-Proyecto/tree/main/PR%C3%81CTICAS* <br>
**URL GitHub (ipynb):** *https://github.com/RGS70/03MIAR-Proyecto/blob/main/PR%C3%81CTICAS/Algoritmo_Genetico_para_el_TSP_Ricardo_Germes_Serrano.ipynb* <br>
**URL GitHub (pdf):** *https://github.com/RGS70/03MIAR-Proyecto/blob/main/PR%C3%81CTICAS/Algoritmo_Genetico_para_el_TSP_Ricardo_Germes_Serrano.pdf* <br>


# Instalación de librerias

In [1]:
#Modulo de llamadas http para descargar ficheros
%pip install requests
import urllib.request

#Libreria del problema TSP: http://elib.zib.de/pub/mp-testdata/tsp/tsplib/tsplib.html
%pip install tsplib95
import tsplib95

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import random   #Libreria para generar numeros y listas aleatorias
import copy     #Permite hacer copias de objetos en python: listas, diccionarios,...
import gzip     #Para descomprimir el fichero .gz sin depender del comando de sistema "gzip"
import shutil
import socket
import os       #Para comprobar si el fichero ya existe en la carpeta local

# Carga de datos del problema

In [3]:
#Librerias y carga del problema

#http://elib.zib.de/pub/mp-testdata/tsp/tsplib/
#Documentacion :
  # https://wwwproxy.iwr.uni-heidelberg.de/groups/comopt/software/TSPLIB95/tsp95.pdf
  # https://tsplib95.readthedocs.io/usage.html
  # https://tsplib95.readthedocs.io/modules.html#module-tsplib95.models

#Limitamos el tiempo de espera de conexión (evita que el notebook se quede
#"colgado" varios minutos si el servidor no responde)
socket.setdefaulttimeout(10)

file = "swiss42.tsp"

#Algunas redes (proxys/firewalls corporativos o de la universidad) no dejan
#conectar con el servidor de la Universidad de Heidelberg. Por eso probamos
#la fuente original y, si falla, comprobamos si el fichero ya está descargado
#en la carpeta local; solo si tampoco está ahí, caemos a un mirror en GitHub
#con los mismos datos (sin necesidad de descomprimir, ya viene en texto plano).
fuente_original = "http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz"
fuente_alternativa = "https://github.com/mastqe/tsplib/raw/refs/heads/master/swiss42.tsp"

descargado = False

#1er intento: fuente original (fichero comprimido en .gz)
try:
  urllib.request.urlretrieve(fuente_original, file + '.gz')
  with gzip.open(file + '.gz', 'rb') as f_in:
    with open(file, 'wb') as f_out:
      shutil.copyfileobj(f_in, f_out)
  descargado = True
  print("Datos descargados desde la fuente original (Universidad de Heidelberg).")
except Exception as e:
  print("Aviso: no se ha podido descargar desde la fuente original ->", e)

#2o intento: comprobamos si el fichero ya existe en la carpeta local
if not descargado and os.path.exists(file):
  descargado = True
  print(f"El fichero '{file}' ya existe en la carpeta local. Se usará ese fichero.")

#3er intento (fallback): mirror alternativo en GitHub
if not descargado:
  try:
    urllib.request.urlretrieve(fuente_alternativa, file)
    descargado = True
    print("Datos descargados desde el mirror alternativo de GitHub.")
  except Exception as e:
    print("Aviso: no se ha podido descargar desde el mirror alternativo ->", e)

if not descargado:
  raise RuntimeError("No ha sido posible descargar el fichero swiss42.tsp desde ninguna fuente.")

#Objeto de tsplib95 para nuestro problema problema
problem = tsplib95.load(file)

#Nodos
Nodos = list(problem.get_nodes())

#Aristas
Aristas = list(problem.get_edges())
print("Número de nodos:", len(Nodos))

#Coordenadas(si estan disponibles en el fichero)
problem.get_display(1)

#Distancia
problem.get_weight(1, 2)

Aviso: no se ha podido descargar desde la fuente original -> <urlopen error timed out>
El fichero 'swiss42.tsp' ya existe en la carpeta local. Se usará ese fichero.
Número de nodos: 42


34

# Funciones de la Actividad Guiada 3

In [4]:
#Funciones de la Actividad Guiada 3
#Se genera una solucion aleatoria con comienzo en en el nodo 0
def crear_solucion(Nodos):
  solucion = [Nodos[0]]
  for n in Nodos[1:] :
    solucion = solucion + [random.choice(list(set(Nodos) - set(solucion)))]
  return solucion

#Devuelve la distancia entre dos nodos
def distancia(a,b, problem):
  return problem.get_weight(a,b)


#Devuelve la distancia total de una trayectoria/solucion
def distancia_total(solucion, problem):
  distancia_total = 0
  for i in range(len(solucion)-1):
    distancia_total += distancia(solucion[i] ,solucion[i+1] ,  problem)
  return distancia_total + distancia(solucion[len(solucion)-1] ,solucion[0], problem)

# Funciones Auxiliares

In [5]:
#Genera una poblacion inicial de soluciones de tamaño N.
# Puede ser válida la solución aleatoria de la AG3: crear_solucion(Nodos)
def generar_poblacion(Nodos,N):
  poblacion = [crear_solucion(Nodos) for _ in range(N)]
  return poblacion

In [6]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
  mejor_solucion = poblacion[0]
  mejor_distancia = distancia_total(poblacion[0], problem)

  for solucion in poblacion[1:]:
    distancia_actual = distancia_total(solucion, problem)
    if distancia_actual < mejor_distancia:
      mejor_solucion = solucion
      mejor_distancia = distancia_actual

  return (mejor_solucion, mejor_distancia)

In [7]:
#Funcion de cruce. Recibe una poblacion(lista de soluciones) y devuelve la población ampliada con los hijos.
# Todos los individuos de la población son selecionados para el cruce(si la población es par)
# Podría aplicarse un proceso previo de selección para elegir los individuos que se desea cruzar.
def Cruzar(poblacion,mutacion, problem):
  #Barajamos la población para emparejar individuos al azar en cada generación
  #(evita cruzar siempre a los mismos vecinos de la lista)
  poblacion_a_cruzar = copy.deepcopy(poblacion)
  random.shuffle(poblacion_a_cruzar)

  hijos = []
  for i in range(0, len(poblacion_a_cruzar) - 1, 2):
    padres = (poblacion_a_cruzar[i], poblacion_a_cruzar[i+1])
    hijos += Descendencia(padres, problem, mutacion)

  #Devolvemos la población ampliada: padres originales + hijos generados
  return poblacion + hijos

In [8]:
#Funcion para generar hijos a partir de 2 padres:
# Se elige el metodo de 1-punto de corte pero es posible usar otros n-puntos, uniforme, dependiendo del problema
def Descendencia(padres, problem,mutacion):
  padre1, padre2 = padres
  n = len(padre1)

  #Elegimos un punto de corte (evitamos la posición 0, que es siempre el nodo de inicio)
  punto_corte = random.randint(1, n-1)

  #Hijo 1: primera parte de padre1 + segunda parte de padre2
  hijo1 = padre1[:punto_corte] + padre2[punto_corte:]
  #Hijo 2: primera parte de padre2 + segunda parte de padre1
  hijo2 = padre2[:punto_corte] + padre1[punto_corte:]

  #Los hijos generados así pueden tener nodos repetidos/faltantes: los reparamos
  hijo1 = Factibilizar(hijo1, problem)
  hijo2 = Factibilizar(hijo2, problem)

  #Aplicamos el operador de mutación a cada hijo
  hijo1 = Mutar(hijo1, mutacion)
  hijo2 = Mutar(hijo2, mutacion)

  return [hijo1, hijo2]

In [9]:
#Para el operador de cruce 1-punto los hijos generados no son soluciones(algunos nodos se repiten y otros no están)
def Factibilizar(solucion,problem):
  Nodos = list(problem.get_nodes())

  #Nodos que faltan por visitar en la solución
  faltantes = [n for n in Nodos if n not in solucion]
  random.shuffle(faltantes)

  vistos = set()
  solucion_factible = []
  for nodo in solucion:
    if nodo not in vistos:
      solucion_factible.append(nodo)
      vistos.add(nodo)
    else:
      #Si el nodo ya apareció antes (repetido), lo sustituimos por uno de los que faltan
      nuevo_nodo = faltantes.pop()
      solucion_factible.append(nuevo_nodo)
      vistos.add(nuevo_nodo)

  return solucion_factible

In [10]:
#Funcion de mutación. Se eligen dos nodos y se intercambia. Se podrian añadir otros operaradores
# Se hace mutaciones mutacion% de las veces
def Mutar(solucion,mutacion):
  if random.random() <= mutacion:
    #Elegimos dos posiciones al azar (evitamos la posición 0, nodo de inicio fijo)
    i, j = random.sample(range(1, len(solucion)), 2)
    solucion[i], solucion[j] = solucion[j], solucion[i]
  return solucion

In [11]:
#Funcion de seleccion de la población. Recibe como parametro una poblacion y
# devuelve una poblacion a la que se ha eliminado individuos poco aptos(fitness alto) y para mantener una poblacion estable de N individuos
#Se tiene en cuenta el porcentaje elitismo pasado como parametro
# Para los individuos que no son de la elite podríamos usar una selección de ruleta(proporcional a su fitness)
def Seleccionar(problem,poblacion, N, elitismo):
  #Ordenamos la población de mejor a peor (menor distancia = mejor)
  poblacion_ordenada = sorted(poblacion, key=lambda solucion: distancia_total(solucion, problem))

  #Nos quedamos siempre con la élite (los mejores individuos)
  n_elite = int(N * elitismo)
  elite = poblacion_ordenada[:n_elite]

  #El resto hasta completar N se elige mediante selección de ruleta, proporcional al fitness
  #(a menor distancia, mayor fitness = mayor probabilidad de ser seleccionado)
  resto_poblacion = poblacion_ordenada[n_elite:]
  distancias = [distancia_total(s, problem) for s in resto_poblacion]
  fitness = [1 / d for d in distancias]
  total_fitness = sum(fitness)
  probabilidades = [f / total_fitness for f in fitness]

  n_restantes = N - n_elite
  seleccionados = random.choices(resto_poblacion, weights=probabilidades, k=n_restantes) if resto_poblacion else []

  return elite + seleccionados

# Proceso Principal

In [12]:
#Funcion principal del algoritmo genetico
#######################################################3
def algoritmo_genetico(problem=problem,N=100,mutacion=.15,elitismo=.1,generaciones=100):
  # problem = datos del problema
  # N = Tamaño de la población
  # mutacion = probabilidad de una mutación
  # elitismo = porcion de la mejor poblacion a mantener
  # generaciones = nº de generaciones a generar para finalizar

  #Genera la poblacion inicial
  Nodos = list(problem.get_nodes())
  poblacion = generar_poblacion(Nodos,N)

  #Inicializamos valores para la mejor solucion
  (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)


  #Condicion de parada
  parar = False
  n=0
  #Inciamos el cliclo de generaciones
  while(parar == False) :

    #Cruce de la poblacion(incluye mutación)
    poblacion = Cruzar(poblacion, mutacion, problem)

    #Seleccionamos la población
    poblacion = Seleccionar(problem,poblacion, N, elitismo)

    #Evaluamos la nueva población
    (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)

    print("Generacion #", n, "\nLa mejor solución es:" , mejor_solucion, "\ncon distancia " , mejor_distancia, "\n")

    #Numero de generaciones. Criterio de parada
    if n==generaciones:
      parar = True
    n +=1

  return mejor_solucion


sol = algoritmo_genetico(problem=problem,N=500,mutacion=.3,elitismo=.40,generaciones=250)

Generacion # 0 
La mejor solución es: [0, 23, 8, 22, 10, 3, 4, 38, 9, 41, 35, 32, 1, 25, 26, 31, 5, 19, 34, 28, 2, 27, 12, 6, 14, 36, 18, 13, 30, 29, 33, 17, 7, 21, 20, 39, 24, 40, 11, 15, 37, 16] 
con distancia  3895 

Generacion # 1 
La mejor solución es: [0, 23, 8, 22, 10, 3, 4, 38, 9, 41, 35, 32, 1, 25, 26, 31, 5, 19, 34, 28, 2, 27, 12, 6, 14, 36, 18, 13, 30, 29, 33, 17, 7, 21, 20, 39, 24, 40, 11, 15, 37, 16] 
con distancia  3895 

Generacion # 2 
La mejor solución es: [0, 35, 24, 40, 9, 8, 2, 3, 32, 7, 39, 20, 27, 11, 22, 5, 15, 17, 36, 31, 38, 25, 26, 12, 10, 30, 21, 34, 1, 4, 18, 29, 28, 41, 23, 19, 33, 37, 14, 13, 6, 16] 
con distancia  3827 

Generacion # 3 
La mejor solución es: [0, 35, 24, 40, 9, 8, 2, 3, 32, 7, 39, 20, 27, 11, 22, 5, 15, 17, 36, 31, 38, 25, 26, 12, 10, 30, 21, 34, 1, 4, 18, 29, 28, 41, 23, 19, 33, 37, 14, 13, 6, 16] 
con distancia  3827 

Generacion # 4 
La mejor solución es: [0, 35, 14, 6, 9, 24, 40, 34, 33, 41, 18, 26, 20, 31, 5, 16, 12, 25, 11, 13, 28, 1